<a href="https://colab.research.google.com/github/SaloneJJ/FHT-Structures-ans-Sperner-Enumerations/blob/main/FHT_enumeration_and_symetry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pynauty networkx

In [ ]:
# FHT Enumeration and Complete Symmetry Analysis by SALONE Jean-Jacques 2026
# jean-jacques.salone@univ-antilles.fr

from itertools import combinations
from collections import Counter
import math
from pynauty import Graph, certificate, autgrp


# ==========================================
# PART 1: Calculation of S(v) with Symmetry
# ==========================================

def get_possible_edges(v):
    """Generates all possible non-empty hyperedges as binary integer bitmasks."""
    edges = []
    for r in range(1, v + 1):
        for comb in combinations(range(v), r):
            edge_mask = 0
            for vertex in comb:
                edge_mask |= (1 << vertex)
            edges.append(edge_mask)
    return edges

def is_sperner_extension(current_edges, new_edge):
    """Checks if adding new_edge respects the Sperner condition (clutter property)."""
    for e in current_edges:
        if (new_edge & e) == new_edge or (new_edge & e) == e:
            return False
    return True

def is_connected_bit(v, edges):
    """Verifies the connectedness of the primal graph using bitmask operations."""
    if not edges:
        return False
    adj = [0] * v
    for e in edges:
        verts = []
        temp = e
        vertex_idx = 0
        while temp:
            if temp & 1:
                verts.append(vertex_idx)
            temp >>= 1
            vertex_idx += 1

        for i in range(len(verts)):
            for j in range(i + 1, len(verts)):
                u, w = verts[i], verts[j]
                adj[u] |= (1 << w)
                adj[w] |= (1 << u)

    visited = 1 << 0
    queue = [0]
    while queue:
        curr = queue.pop(0)
        neighbors = adj[curr]
        unvisited = neighbors & ~visited
        while unvisited:
            bit = unvisited & -unvisited
            next_v = bit.bit_length() - 1
            visited |= bit
            queue.append(next_v)
            unvisited ^= bit

    return visited == ((1 << v) - 1)

def get_pynauty_graph(v, edges):
    """Constructs a bipartite graph representing the hypergraph structure for pynauty."""
    m = len(edges)
    n_nodes = v + m
    adj = {i: [] for i in range(n_nodes)}

    vertex_class = set(range(v))
    edge_class = set(range(v, v + m))

    for idx, e in enumerate(edges):
        edge_node = v + idx
        temp = e
        u = 0
        while temp:
            if temp & 1:
                adj[u].append(edge_node)
                adj[edge_node].append(u)
            temp >>= 1
            u += 1

    g = Graph(
        number_of_vertices=n_nodes,
        directed=False,
        adjacency_dict=adj,
        vertex_coloring=[vertex_class, edge_class]
    )
    return g

def extract_symmetry_metadata(v, edges):
    """Helper function to extract Aut size, orbits, and fixed points from a hypergraph edge set."""
    g = get_pynauty_graph(v, edges)
    aut_data = autgrp(g)
    grpsize1 = aut_data[1]
    grpsize2 = aut_data[2]
    orbits = aut_data[3]
    num_orbits = aut_data[4]

    aut_size = grpsize1 * (10 ** grpsize2)
    vertex_orbits = orbits[:v]
    fixed_points_count = sum(1 for node_val in vertex_orbits if vertex_orbits.count(node_val) == 1)

    return {
        "edges": list(edges),
        "aut_size": aut_size,
        "orbits": num_orbits,
        "inv_points": fixed_points_count
    }

def compute_S_v_with_symmetry(v):
    """Computes S(v) and extracts symmetry parameters for flat structures."""
    if v == 1:
        return 1, [extract_symmetry_metadata(1, [1])]
    if v == 2:
        return 1, [extract_symmetry_metadata(2, [3])] # edge {0,1} -> mask 3

    all_edges = get_possible_edges(v)
    seen_certificates = set()
    structures_metadata = []

    def backtrack(edge_index, current_hg):
        if current_hg:
            g = get_pynauty_graph(v, current_hg)
            cert = certificate(g)
            if cert in seen_certificates:
                return
            seen_certificates.add(cert)

            if is_connected_bit(v, current_hg):
                meta = extract_symmetry_metadata(v, current_hg)
                structures_metadata.append(meta)

        for i in range(edge_index, len(all_edges)):
            candidate = all_edges[i]
            if is_sperner_extension(current_hg, candidate):
                current_hg.append(candidate)
                backtrack(i + 1, current_hg)
                current_hg.pop()

    backtrack(0, [])
    return len(structures_metadata), structures_metadata


# ==========================================
# PART 2: Decomposition & Symmetry for I(v)
# ==========================================

def generate_valid_partitions(v):
    """Generates partitions of v according to rules: v_1 > 1, v_k > 0, and v_k < v."""
    if v <= 2:
        return []

    partitions = []
    def backtrack_part(remaining, max_val, current):
        if remaining == 0:
            if len(current) > 0 and current[0] > 1:
                partitions.append(tuple(current))
            return
        start = min(remaining, max_val)
        for i in range(start, 0, -1):
            if i < v:
                backtrack_part(remaining - i, i, current + [i])

    backtrack_part(v, v - 1, [])
    seen = set()
    valid = []
    for p in partitions:
        if p[0] > 1 and all(x < v for x in p) and p not in seen:
            seen.add(p)
            valid.append(p)
    return valid

def compute_I_v_with_symmetry(v, all_fht_details, f_dict):
    """
    Computes I(v) and analyzes the symmetries of the resulting imbricated structures
    by explicitly reconstructing the global sum hypergraph for each valid composition.
    """
    if v <= 2:
        return 0, []

    partitions = generate_valid_partitions(v)
    imbricated_metadata = []
    total_count = 0

    for p in partitions:
        # p represents block sizes, e.g., (2, 2) or (3, 1) etc.
        # To generate all unique combinations of sub-FHTs corresponding to partition p:
        # We need to map block sizes to actual sub-FHT edge structures from our catalog.

        # Helper to recursively build combinations of sub-FHT choices for this partition structure
        def build_compositions(part_idx, current_sub_fhts_meta, current_vertex_offset):
            nonlocal total_count
            if part_idx == len(p):
                # We have selected a specific combination of sub-FHTs for the partition blocks.
                # Now we apply the sum operation over the global vertex set [0, ..., v-1].
                # 1. Gather all internal edges with shifted vertex indices
                global_edges = []
                offset = 0
                for block_size, sub_fht in zip(p, current_sub_fhts_meta):
                    for edge_mask in sub_fht["edges"]:
                        # Shift the bitmask bits to align with the current block's global vertex range
                        shifted_mask = edge_mask << offset
                        global_edges.append(shifted_mask)
                    offset += block_size

                # 2. Add the global root edge V (which is (1 << v) - 1)
                global_root_edge = (1 << v) - 1
                global_edges.append(global_root_edge)

                # 3. Extract symmetry metadata for this newly formed imbricated FHT structure
                meta = extract_symmetry_metadata(v, global_edges)
                imbricated_metadata.append(meta)
                total_count += 1
                return

            block_size = p[part_idx]
            # Get all stored FHT metadata for this block size (both S and I combined if available,
            # or we fetch from our global catalog of computed FHTs for size block_size)
            available_fhts = all_fht_details[block_size]

            # To account for multiset combinations correctly without generating redundant label permutations,
            # we can iterate or combine. For symmetry classification of classes, let's explore representative choices.
            for sub_fht in available_fhts:
                build_compositions(part_idx + 1, current_sub_fhts_meta + [sub_fht], current_vertex_offset + block_size)

        build_compositions(0, [], 0)

    return total_count, imbricated_metadata


# ==========================================
# MAIN SCRIPT WITH COMPLETE SYMMETRY REPORT
# ==========================================

if __name__ == "__main__":
    try:
        max_v = int(input("Enter the maximum value of v to calculate: "))
        if max_v < 1:
            print("Please enter an integer greater than or equal to 1.")
            exit()
    except ValueError:
        print("Invalid input.")
        exit()

    f = {}
    S = {}
    I = {}
    all_fht_details = {} # Stores all structures (Flat S + Imbricated I) with their metadata for each v

    print(f"\n--- Starting complete FHT & Symmetry analysis from v = 1 to {max_v} ---\n")

    for v in range(1, max_v + 1):
        # Step 1: Compute S(v) and its symmetries
        if v == 1:
            S[v] = 1
            s_details = [extract_symmetry_metadata(1, [1])]
        elif v == 2:
            S[v] = 1
            s_details = [extract_symmetry_metadata(2, [3])]
        else:
            print(f"[{v}] Computing S({v}) flat structures & symmetries...")
            S[v], s_details = compute_S_v_with_symmetry(v)

        # Step 2: Compute I(v) and its symmetries via partition sum reconstruction
        if v <= 2:
            I[v] = 0
            i_details = []
        else:
            print(f"[{v}] Computing I({v}) hierarchical structures & symmetries...")
            # Note: all_fht_details must contain all FHTs for sizes < v
            I[v], i_details = compute_I_v_with_symmetry(v, all_fht_details, f)

        # Step 3: Deduce total f(v) and merge full catalogue for this size v
        f[v] = S[v] + I[v]
        all_fht_details[v] = s_details + i_details

        print(f"--> Results for v = {v}: S({v}) = {S[v]} | I({v}) = {I[v]} | f({v}) = {f[v]}")

        # Display breakdown of symmetry profiles for all structures at size v
        print(f"    Symmetry profiles for all {f[v]} FHT structures of size {v}:")
        for idx, meta in enumerate(all_fht_details[v], 1):
            struct_type = "Flat (S)" if idx <= S[v] else "Imbricated (I)"
            print(f"      [{struct_type}] # {idx}: |Aut(H)| = {meta['aut_size']:<5} | Orbits (k) = {meta['orbits']:<3} | Invariant Points = {meta['inv_points']}")
        print("-" * 70)